# Trening w Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MartaL-ctrl/Counting-crops/blob/main/notebooks/colab_train.ipynb)

**Jak używać:**
1. *Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU (np. T4)*.
2. W komórce **Ustawienia** wybierz plik konfiguracyjny (`CONFIG`).
3. *Środowisko wykonawcze → Uruchom wszystko*. Za pierwszym razem Colab poprosi o dostęp do Dysku Google.

**Gdzie co jest:**
- kod: GitHub → `/content/Counting-crops` (przy każdym uruchomieniu `git pull`),
- dane: Dysk Google `MyDrive/counting-crops/data/` → kopia na lokalnym dysku Colaba `/content/data/gwhd_hf`,
- wyniki: Dysk Google `MyDrive/counting-crops/outputs/<run_name>/`.

**Po rozłączeniu sesji** uruchom wszystko ponownie. Jeśli na Dysku jest `last.pt`, trening wznowi się
od następnej epoki.

Kod zmieniamy lokalnie w VS Code (commit + push), a nie w Colabie: zmiany w `/content` znikają po sesji.

## Ustawienia

In [ ]:
CONFIG = "configs/colab_smoke.yaml"   # <- plik konfiguracyjny z repozytorium
RESUME = True                         # wznów od last.pt, jeśli istnieje

REPO_URL = "https://github.com/MartaL-ctrl/Counting-crops.git"
REPO_DIR = "/content/Counting-crops"
DRIVE_DIR = "/content/drive/MyDrive/counting-crops"   # dane i wyniki na Dysku Google
LOCAL_DATA = "/content/data/gwhd_hf"                  # musi się zgadzać z data_dir w konfiguracji

# Zbiór danych: przypięta wersja (commit) repozytorium na Hugging Face -> zawsze te same pliki
HF_REPO = "GlobalWheat/GWHD_2021"
HF_REVISION = "976d2f12ca59bff731d3e54713d77bf9aa3bca03"
HF_FILES = [
    "data/train-00000-of-00002.parquet",
    "data/train-00001-of-00002.parquet",
    "data/validation-00000-of-00001.parquet",
    "data/test-00000-of-00001.parquet",
]

## 1. Dysk Google

Daje Colabowi dostęp do Twojego Dysku jako folderu `/content/drive/MyDrive`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Kod z GitHuba

Pierwsze uruchomienie w sesji: `git clone`; kolejne: `git pull`. Wypisany skrót commita warto zanotować -
mówi, która wersja kodu dała dany wynik (zapisuje się też do `outputs/<run_name>/git_commit.txt`).

In [ ]:
import os
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
os.chdir(REPO_DIR)
print("commit:", COMMIT, "| katalog roboczy:", os.getcwd())

## 3. GPU i biblioteki

PyTorch, torchvision, pandas, pyarrow i PyYAML są w Colabie preinstalowane - **nie instalujemy ich ponownie**
(nadpisanie PyTorch może zepsuć obsługę GPU). Komórka tylko sprawdza wersje.

Jeśli `GPU: False` - zmień typ środowiska wykonawczego na GPU i uruchom wszystko od nowa.

In [ ]:
import sys

import pandas, pyarrow, torch, torchvision, yaml

print("Python", sys.version.split()[0])
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| pandas", pandas.__version__, "| pyarrow", pyarrow.__version__, "| pyyaml", yaml.__version__)
print("GPU:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
assert torch.cuda.is_available(), "Brak GPU - zmień typ środowiska wykonawczego"

## 4. Dane

1. Jeśli plików nie ma na Dysku - **jednorazowo** pobiera je z Hugging Face na Dysk (ok. 1,3 GB, kilka minut).
2. Kopiuje je z Dysku na lokalny dysk Colaba - odczyt w trakcie treningu jest wtedy dużo szybszy
   niż bezpośrednio z Dysku (ok. 1 min na sesję).

In [ ]:
import shutil
from pathlib import Path

from huggingface_hub import hf_hub_download

drive_data = Path(DRIVE_DIR)          # pliki trafią do DRIVE_DIR/data/*.parquet
local_data = Path(LOCAL_DATA)
local_data.mkdir(parents=True, exist_ok=True)

for name in HF_FILES:
    src = drive_data / name
    if not src.exists():
        print("pobieranie z Hugging Face:", name)
        hf_hub_download(HF_REPO, name, repo_type="dataset", revision=HF_REVISION, local_dir=drive_data)
    dst = local_data / Path(name).name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
    print(f"{dst.name}: {dst.stat().st_size / 1e6:.0f} MB")

## 5. Trening

Wynik każdej epoki pojawia się na bieżąco. Jeśli `RESUME = True` i istnieje `last.pt`, trening jest wznawiany.

Colab rozłącza sesję po dłuższej bezczynności karty przeglądarki i po kilku godzinach pracy.
Checkpoint `last.pt` zapisuje się po każdej epoce, więc tracisz najwyżej jedną epokę.

In [ ]:
import yaml

with open(CONFIG, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
assert cfg["data_dir"] == LOCAL_DATA, f"data_dir w {CONFIG} ({cfg['data_dir']}) != LOCAL_DATA ({LOCAL_DATA})"

run_dir = Path(cfg["output_dir"]) / cfg["run_name"]
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / "git_commit.txt").write_text(COMMIT + "\n")

last = run_dir / "last.pt"
resume_args = f"--resume {last}" if RESUME and last.exists() else ""
print("wyniki:", run_dir, "|", "wznowienie od " + str(last) if resume_args else "nowy trening")

!python train.py --config {CONFIG} {resume_args}

## 6. Podgląd wyników

Pliki zostają na Dysku Google: `MyDrive/counting-crops/outputs/<run_name>/`.

In [ ]:
import pandas as pd

hist = pd.read_csv(run_dir / "history.csv")
display(hist)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["epoch"], hist["train_loss"])
ax[0].set(title="strata treningowa", xlabel="epoka", yscale="log")
ev = hist.dropna(subset=["val_mae"])
ax[1].plot(ev["epoch"], ev["val_mae"], "o-", label="validation")
if ev["train_mae"].notna().any():
    ax[1].plot(ev["epoch"], ev["train_mae"], "o-", label="train")
ax[1].set(title="MAE liczby kłosów", xlabel="epoka")
ax[1].legend()
plt.show()